# Phase 5 — RL Pit Agent (PPO) — Kaggle
Trains the PPO pit-strategy agent inside the race simulator (`src/rl/env.py`),
across **3 seeds**, then runs the head-to-head versus the Phase 4a MDP policy in the
SAME simulator (`src/rl/evaluate_rl.py`). A documented "RL matched but did not beat the
MDP" is a legitimate result (MASTER_CONTEXT 10.5) — this notebook reports it either way.

## Honest note on the hardware (READ THIS)
**PPO with an MLP policy is mostly CPU-bound.** The forward/backward passes on a tiny
MLP are dwarfed by the Python env-stepping, which runs on CPU. The 2x T4 GPUs do **not**
meaningfully speed up this job. The sensible use of the hardware here is:
- **`SubprocVecEnv` with `--n-envs 8`** — parallel env rollouts across CPU cores (the real
  throughput lever). Kaggle gives ~4 vCPUs, so 8 envs oversubscribe slightly, which is fine.
- **Run the 3 seeds sequentially** in one session (default below), OR launch two seeds as
  separate processes pinned one per GPU (`--device cuda`) if and only if a GPU run profiles
  faster on your config — usually it does not for an MLP. Keep `--device cpu` unless measured.

## Budget (~10 h total, per the playbook)
3 seeds x 2-5M steps, checkpoints every 500k. At ~300-400 env-steps/s on CPU with 8 envs,
~3M steps is ~2.5-3.5 h per seed -> ~8-10 h for 3 seeds. Tune `STEPS` to your session budget
(Kaggle caps a session at **12 h**). Checkpoints every 500k mean a dead session loses < 30 min.

## Before running
1. Notebook settings: **Accelerator = GPU T4 x2** (for parity with other runs; PPO uses CPU
   unless you pass `--device cuda`), **Internet = ON**, **Persistence = Files only** (so
   checkpoints survive a restart).
2. **+ Add Input** -> attach the `tft_full_data` dataset (the `laps_*_r*.parquet` files,
   same dataset as notebooks 04-07). The env needs `models/tyre_curves.joblib`, which we
   regenerate from these parquets in cell 4 (fast, CPU).
3. Repo pushed to GitHub `main` (cell 2 clones/pulls it). `data/pit_loss.json`,
   `models/tft_calibration.json`, and `reports/lap_time/tft_recalibration.csv` are committed,
   so they arrive with the clone.

**When done:** download `rl_ppo_artifacts.zip` from the Output panel (models + tb logs +
eval csv). Local steps in the last cell.

In [1]:
# 1. Deps: stable-baselines3 + gymnasium (Section 3 RL block). Keep Kaggle's stock torch.
!pip -q install stable-baselines3 gymnasium tensorboard fastf1 pandera scipy joblib
!pip install mlflow
import stable_baselines3 as sb3, gymnasium as gym, torch
print('sb3', sb3.__version__, '| gymnasium', gym.__version__, '| torch', torch.__version__)
print('cuda:', torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.9/386.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 78.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 

2026-06-12 08:28:35.665958: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781252916.147412      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781252916.274192      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781252917.362564      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781252917.362608      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781252917.362611      16 computation_placer.cc:177] computation placer alr

sb3 2.8.0 | gymnasium 1.2.0 | torch 2.10.0+cpu
cuda: False


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# 2. Clone repo (or pull) + path
import os, sys
REPO = '/kaggle/working/f1-strategist'
if not os.path.exists(REPO):
    !git clone https://github.com/Shreyansh262/f1-strategist.git $REPO
else:
    !cd $REPO && git pull
sys.path.insert(0, REPO)
os.chdir(REPO)
!cd $REPO && git log --oneline -1

Cloning into '/kaggle/working/f1-strategist'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (434/434), done.
remote: Compressing objects: 100% (290/290), done.
remote: Total 434 (delta 182), reused 371 (delta 119), pack-reused 0 (from 0)
Receiving objects: 100% (434/434), 5.51 MiB | 27.95 MiB/s, done.
Resolving deltas: 100% (182/182), done.
527f8e2 (HEAD -> main, origin/main, origin/HEAD) RL implementation+HF deployment


In [3]:
# 3. Copy parquets from the attached dataset into data/raw (same as runs 04-07)
import glob, shutil, pathlib
dst = pathlib.Path(REPO) / 'data' / 'raw'
dst.mkdir(parents=True, exist_ok=True)
src_files = glob.glob('/kaggle/input/**/laps_*_r*.parquet', recursive=True)
assert src_files, 'No laps_*_r*.parquet under /kaggle/input/ — attach the data dataset first.'
for f in src_files:
    shutil.copy(f, dst / pathlib.Path(f).name)
copied = sorted(p.name for p in dst.glob('laps_*_r*.parquet'))
print(len(copied), 'files | seasons:', sorted({n.split('_')[1] for n in copied}))

94 files | seasons: ['2022', '2023', '2024', '2025', '2026']


In [4]:
# 4. Ensure the env's model inputs exist.
#    The RaceEnv needs models/tyre_curves.joblib (gitignored). Regenerate it from the
#    parquets via the Phase 3 fit (fast, CPU). pit_loss.json / tft_calibration.json /
#    tft_recalibration.csv are committed and arrived with the clone.
import pathlib
curves = pathlib.Path(REPO) / 'models' / 'tyre_curves.joblib'
if not curves.exists():
    print('Fitting tyre curves (one-off, ~1-2 min)...')
    from src.models.tyre.fit import fit as fit_tyres
    fit_tyres()
assert curves.exists(), 'tyre_curves.joblib still missing'
# sanity: build one env
from src.rl.env import RaceEnv
from gymnasium.utils.env_checker import check_env
check_env(RaceEnv(), skip_render_check=True)
print('env OK — check_env passed')

Fitting tyre curves (one-off, ~1-2 min)...


/usr/local/lib/python3.12/dist-packages/pandera/_pandas_deprecated.py:143: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)
08:29:13  INFO      Loaded 94 files | 103732 rows -> 103732 after dedup
08:29:13  INFO      Removed 5147 pit out-laps (TyreLife == 1)
08:29:13  WARNING   Dropping 3640 

env OK — check_env passed


/usr/local/lib/python3.12/dist-packages/gymnasium/utils/env_checker.py:311: UserWarning: WARN: A Box observation space minimum value is -infinity. This is probably too low.
  logger.warn(
/usr/local/lib/python3.12/dist-packages/gymnasium/utils/env_checker.py:315: UserWarning: WARN: A Box observation space maximum value is infinity. This is probably too high.
  logger.warn(


## Train — 3 seeds (sequential, parallel envs)
Each seed: PPO MlpPolicy, `n_envs=8` (SubprocVecEnv), checkpoints every 500k to
`models/rl/`. Set `STEPS` to fit your session. The trainer is resumable in spirit —
if a session dies, completed seeds' final `.zip` files persist (Persistence = Files only);
re-run this cell and skip seeds whose final artifact already exists.

In [5]:
# 5. Train the 3 seeds
from pathlib import Path
from src.rl.train_ppo import train, MODELS_DIR

STEPS  = 3_000_000     # 2-5M per the playbook; 3M ~ a sensible mid-point
N_ENVS = 8             # parallel env rollouts (the real throughput lever on CPU)
SEEDS  = [0, 1, 2]
DEVICE = 'cpu'         # MLP PPO is CPU-bound; only flip to 'cuda' if you profile a win

for seed in SEEDS:
    final = MODELS_DIR / f'ppo_pit_seed{seed}.zip'
    if final.exists():
        print(f'seed {seed}: final artifact exists, skipping'); continue
    print(f'=== training seed {seed} ===', flush=True)
    train(steps=STEPS, seed=seed, n_envs=N_ENVS, device=DEVICE, checkpoint_freq=500_000)
print('All seeds done. Artifacts in', MODELS_DIR)

=== training seed 0 ===


2026-06-12 08:29:33.221868: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 08:29:33.258913: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 08:29:33.303444: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 08:29:33.319943: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781252973.325333     110 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already b

Using cpu device


08:30:07  INFO      PPO training: steps=3000000 seed=0 n_envs=8 device=cpu ckpt_freq=500000


Logging to /kaggle/working/f1-strategist/models/rl/tb/seed0_1
------------------------------
| time/              |       |
|    fps             | 880   |
|    iterations      | 1     |
|    time_elapsed    | 18    |
|    total_timesteps | 16384 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 830         |
|    iterations           | 2           |
|    time_elapsed         | 39          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.009887422 |
|    clip_fraction        | 0.0639      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | -1.33       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.00651    |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00323    |
|    value_loss           | 0.251       |
-------

...

=== training seed 2 ===


2026-06-12 10:38:02.072763: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-06-12 10:38:02.158264: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781260682.167206     400 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-06-12 10:38:02.171704: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781260682.199339     400 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-06-12 10:38:

Using cpu device
Logging to /kaggle/working/f1-strategist/models/rl/tb/seed2_1
------------------------------
| time/              |       |
|    fps             | 919   |
|    iterations      | 1     |
|    time_elapsed    | 17    |
|    total_timesteps | 16384 |
------------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 841         |
|    iterations           | 2           |
|    time_elapsed         | 38          |
|    total_timesteps      | 32768       |
| train/                  |             |
|    approx_kl            | 0.013754821 |
|    clip_fraction        | 0.121       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.38       |
|    explained_variance   | -1.21       |
|    learning_rate        | 0.0003      |
|    loss                 | -0.0154     |
|    n_updates            | 10          |
|    policy_gradient_loss | -0.00377    |
|    value_loss           | 0.20

11:42:10  INFO      Saved final policy -> /kaggle/working/f1-strategist/models/rl/ppo_pit_seed2.zip


All seeds done. Artifacts in /kaggle/working/f1-strategist/models/rl


## Evaluate — PPO vs MDP, paired (common random numbers)
Both policies roll out the same RaceEnv episodes under identical seeds, so the
difference is the policy, not luck. We evaluate each seed's final policy; the
best-by-mean-finish seed is the headline.

In [6]:
# 6. Head-to-head for each seed
import pandas as pd
from src.rl.evaluate_rl import evaluate, REPORTS_DIR

EPISODES = 500
summary = []
for seed in SEEDS:
    model = MODELS_DIR / f'ppo_pit_seed{seed}.zip'
    if not model.exists():
        print(f'seed {seed}: no model, skipping'); continue
    out = evaluate(model, episodes=EPISODES)
    out.to_csv(REPORTS_DIR / f'ppo_vs_mdp_seed{seed}.csv', index=False)
    agg = out[out['seed'] == 'AGGREGATE'].iloc[0]
    summary.append({'seed': seed, 'ppo_mean_finish': agg['ppo_finish'],
                    'mdp_mean_finish': agg['mdp_finish'],
                    'ppo_minus_mdp_time_s': agg['ppo_minus_mdp_time_s'],
                    'ppo_legal_frac': agg['ppo_legal']})
summary = pd.DataFrame(summary)
summary.to_csv(REPORTS_DIR / 'ppo_vs_mdp.csv', index=False)
print(summary.to_string(index=False))
print('\nReading: negative ppo_minus_mdp_time_s = PPO faster than the MDP (a beat).')

11:43:27  INFO      ppo_vs_mdp.csv -> /kaggle/working/f1-strategist/reports/rl/ppo_vs_mdp.csv
11:43:27  INFO      Paired 500 eps: mean PPO-MDP time = +1.501s (PPO faster in 48.6% of races). PPO legal 100.0% / MDP legal 100.0%
11:44:44  INFO      ppo_vs_mdp.csv -> /kaggle/working/f1-strategist/reports/rl/ppo_vs_mdp.csv
11:44:44  INFO      Paired 500 eps: mean PPO-MDP time = +2.448s (PPO faster in 41.2% of races). PPO legal 100.0% / MDP legal 100.0%
11:46:04  INFO      ppo_vs_mdp.csv -> /kaggle/working/f1-strategist/reports/rl/ppo_vs_mdp.csv
11:46:04  INFO      Paired 500 eps: mean PPO-MDP time = -0.354s (PPO faster in 51.0% of races). PPO legal 100.0% / MDP legal 100.0%


 seed  ppo_mean_finish  mdp_mean_finish  ppo_minus_mdp_time_s  ppo_legal_frac
    0            2.574            2.664              1.500755             1.0
    1            2.598            2.664              2.447703             1.0
    2            2.576            2.664             -0.353640             1.0

Reading: negative ppo_minus_mdp_time_s = PPO faster than the MDP (a beat).


In [7]:
# 7. Bundle artifacts for download (Output panel): models + tb logs + eval csvs.
from pathlib import Path
(Path(REPO) / 'models' / 'rl').mkdir(parents=True, exist_ok=True)
(Path(REPO) / 'reports' / 'rl').mkdir(parents=True, exist_ok=True)
!cd $REPO && zip -qr /kaggle/working/rl_ppo_artifacts.zip models/rl reports/rl
print('Download: /kaggle/working/rl_ppo_artifacts.zip')
!ls -la $REPO/models/rl $REPO/reports/rl

Download: /kaggle/working/rl_ppo_artifacts.zip
/kaggle/working/f1-strategist/models/rl:
total 3204
drwxr-xr-x 3 root root   4096 Jun 12 11:42 .
drwxr-xr-x 3 root root   4096 Jun 12 08:29 ..
-rw-r--r-- 1 root root      0 Jun 12 08:29 .gitkeep
-rw-r--r-- 1 root root 151709 Jun 12 08:50 ppo_pit_seed0_1000000_steps.zip
-rw-r--r-- 1 root root 151709 Jun 12 09:01 ppo_pit_seed0_1500000_steps.zip
-rw-r--r-- 1 root root 151710 Jun 12 09:11 ppo_pit_seed0_2000000_steps.zip
-rw-r--r-- 1 root root 151711 Jun 12 09:22 ppo_pit_seed0_2500000_steps.zip
-rw-r--r-- 1 root root 151713 Jun 12 09:32 ppo_pit_seed0_3000000_steps.zip
-rw-r--r-- 1 root root 151697 Jun 12 08:40 ppo_pit_seed0_500000_steps.zip
-rw-r--r-- 1 root root 151713 Jun 12 09:33 ppo_pit_seed0.zip
-rw-r--r-- 1 root root 151713 Jun 12 09:54 ppo_pit_seed1_1000000_steps.zip
-rw-r--r-- 1 root root 151713 Jun 12 10:05 ppo_pit_seed1_1500000_steps.zip
-rw-r--r-- 1 root root 151714 Jun 12 10:16 ppo_pit_seed1_2000000_steps.zip
-rw-r--r-- 1 root root 

## After downloading (LOCAL steps)
1. Unzip `rl_ppo_artifacts.zip` into the repo root (populates `models/rl/*.zip` and
   `reports/rl/*.csv`; model binaries stay gitignored, the report CSVs get committed).
2. Sanity-check locally: `python -m src.rl.evaluate_rl --model models/rl/ppo_pit_seed0.zip --episodes 200`.
3. `pytest -q` — all green (RL env tests don't need the trained model).
4. Write up the result honestly in the model card + MASTER_CONTEXT Section 8 —
   whether PPO beat, matched, or lost to the MDP, with the paired-comparison numbers.